# OT PCAP Triage Tool -- Full Automated Run

No install required. Run each cell below in order (click the play button on the left of each cell, or press Shift+Enter).

This version:
1. Installs the tool
2. Asks you to upload a `.pcap` file
3. Optionally asks for a Fenceline `advisories.json` (skip if you don't have one)
4. Runs the full analysis, automatically cross-referencing Fenceline if you provided it
5. Downloads two PDFs: the full technical report, and a branded executive summary

In [ ]:
# Step 1: Setup (run this first -- takes about 30-60 seconds)
!pip install -q "ot-pcap-triage[pdf] @ git+https://github.com/tjackson8817/OT-PCAP-Triage.git"
print("Setup complete.")

In [ ]:
# Step 2: Upload your capture file
from google.colab import files
print("Choose your .pcap or .pcapng file:")
uploaded = files.upload()
pcap_filename = list(uploaded.keys())[0]
print(f"Uploaded: {pcap_filename}")

In [ ]:
# Step 3: Optional -- upload a Fenceline advisories.json to cross-reference
# Run this cell, then either upload a file OR just click Cancel/press Escape to skip it.
advisories_filename = None
print("If you have a Fenceline advisories.json, upload it now. Otherwise, click Cancel to skip.")
try:
    adv_uploaded = files.upload()
    if adv_uploaded:
        advisories_filename = list(adv_uploaded.keys())[0]
        print(f"Uploaded: {advisories_filename} -- will cross-reference automatically.")
except Exception:
    pass
if not advisories_filename:
    print("Skipped -- running without Fenceline cross-reference.")

In [ ]:
# Step 4: A few optional details for the executive report (edit these, then run)
client_name = "[Client Name]"      #@param {type:"string"}
site_scope = "[Site / Scope]"       #@param {type:"string"}
prepared_by = "Thomas Jackson, Principal"  #@param {type:"string"}
firm_name = "Sale Fish Marketing & Consulting"  #@param {type:"string"}
print("Details set. Continue to the next cell.")

In [ ]:
# Step 5: Run everything and generate both PDFs
import json
from IPython.display import Markdown, display
from ot_pcap_triage import triage, render_markdown, render_html, render_exec_html
from ot_pcap_triage.pdf import html_to_pdf
from ot_pcap_triage.report_html import markdown_bold_to_html
from ot_pcap_triage.report_md import render_executive_summary

crossref_result = None
crossref_md = None
if advisories_filename:
    from ot_pcap_triage import cross_reference, render_crossref_markdown
    findings_preview = triage(pcap_filename)
    payload = findings_preview.as_dict()
    with open(advisories_filename) as f:
        advisories = json.load(f)
        if isinstance(advisories, dict):
            advisories = advisories.get("advisories", [])
    crossref_result = cross_reference(payload, advisories)
    crossref_md = render_crossref_markdown(crossref_result)

findings = triage(pcap_filename)
report_md, summary_rows = render_markdown(findings, pcap_filename, crossref_md=crossref_md)

display(Markdown(report_md))

# Technical report -> PDF
exec_text = render_executive_summary(summary_rows, pcap_filename, 0)
exec_lines = [l for l in exec_text.split("\n") if l.strip() and not l.strip().startswith("#")]
exec_html_lines = "<br>".join(markdown_bold_to_html(l) for l in exec_lines)
html_report = render_html(findings, pcap_filename, summary_rows, exec_summary_text=exec_html_lines)
tech_pdf_path = pcap_filename + ".report.pdf"
html_to_pdf(html_report, tech_pdf_path)

# Executive summary -> PDF
payload = findings.as_dict()
exec_html = render_exec_html(payload, client=client_name, site=site_scope,
                              prepared_by=prepared_by, firm=firm_name,
                              crossref_result=crossref_result)
exec_pdf_path = pcap_filename + ".executive_summary.pdf"
html_to_pdf(exec_html, exec_pdf_path)

print("\n\nDownloading both PDFs now...")
files.download(tech_pdf_path)
files.download(exec_pdf_path)

---
*Nothing you upload here is stored anywhere permanent -- this notebook runs in a temporary, private Google Colab session that's discarded when you close the tab. Built by Thomas Jackson, Sale Fish Marketing & Consulting.*